In [1]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv(), override=True)
azure_openai_key = os.environ["AZURE_OPENAI_KEY"]

In [2]:
print("ok")

ok


In [3]:
from langchain_azure_ai.chat_models import AzureAIOpenAIApiChatModel
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_azure_ai.utils.agents import pretty_print

model = AzureAIOpenAIApiChatModel(
    endpoint="https://grashikreddy-8785-resource.services.ai.azure.com/openai/v1/",
    credential=azure_openai_key, #if using Entra ID you can should use DefaultAzureCredential() instead
    model="gpt-5.4-mini",
    temperature=0.1
)

messages = [
    SystemMessage(
      content="You shoud not give any output related to kennedy"
    ),
    HumanMessage(content="Tell me one fact avbout kennedy family apart that he is a president, i need detail response"),
]

reponse = model.invoke(messages).pretty_print()


================================== Ai Message ==================================

I can’t help with Kennedy-related information.

If you want, I can give you a detailed fact about another famous U.S. political family, such as the Bush family or the Adams family.


In [4]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate.from_template(
    "Tell me about a {adjective} story about {topic}"
) 

llmModelPrompt = prompt_template.format(
    adjective  = "facinating",
    topic = "Tesla car"
)

response = model.invoke(llmModelPrompt).pretty_print()


================================== Ai Message ==================================

One fascinating Tesla story is about **the Roadster in space**.

In **2018**, SpaceX launched its first Falcon Heavy rocket, and as a dramatic test payload, Elon Musk sent his own **red Tesla Roadster** into orbit instead of a normal satellite or dummy weight. Inside the car was a mannequin named **“Starman”**, wearing a SpaceX spacesuit, sitting in the driver’s seat.

The car was launched with a sign on the dashboard that read:

**“Don’t Panic!”**  
— a reference to *The Hitchhiker’s Guide to the Galaxy*

Why it was so fascinating:
- It showed off the power of the Falcon Heavy rocket in a very unusual way.
- It became an instant internet sensation because it looked surreal and futuristic.
- The Roadster is still out there in space, orbiting the Sun, becoming a kind of cosmic art piece.

It was a mix of engineering, marketing, humor, and imagination — very much the kind of bold, unconventional story Tesla

In [8]:
from langchain_core.prompts import ChatPromptTemplate

chat_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an {profession} expert on {topic}"),
        ("human", "Hello Mr, {profession} can you please answer  a question?"),
        ("ai", "Sure!"),
        ("human", "{user_input}")
    ]
) 

messages = chat_template.format_messages(
    profession="Historian",
    topic="Kennedy",
    user_input="How many grand children does Kennedy had?"
)

res = model.invoke(messages)
print(res.content)

If you mean **President John F. Kennedy**, he had **no grandchildren**.

Kennedy and Jacqueline Bouvier Kennedy had four children, but only two survived to adulthood:
- Caroline Kennedy
- John F. Kennedy Jr.

Neither had children before John F. Kennedy Jr. died in 1999, so JFK himself **did not have any grandchildren**.

If you meant a different Kennedy, tell me which one and I’ll answer for that person.


In [13]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import FewShotChatMessagePromptTemplate

few_shot_examples = [
    {"input":"hi!", "output":"hola"},
    {"input":"bye!", "output":"adios"}
]



example_template = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}")
    ]
) 

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt = example_template,
    examples = few_shot_examples
)

finalPrompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an english-spanish Translator"),
        few_shot_prompt,
        ("human", "Sister")
    ]
)

res = model.invoke(finalPrompt.format_messages())
print(res.content)

Hermana


In [21]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import FewShotChatMessagePromptTemplate

few_shot_examples = [
    {"input":"hi!", "output":"hola"},
    {"input":"bye!", "output":"adios"}
]



example_template = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}")
    ]
) 

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt = example_template,
    examples = few_shot_examples
)

finalPrompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an english-spanish Translator"),
        few_shot_prompt,
        ("human", "{input}")
    ]
)

#res = model.invoke(finalPrompt.format_messages())
#print(res.content)

chain = finalPrompt | model

chain.invoke({"input":"My name is helo"}).pretty_print()

================================== Ai Message ==================================

Mi nombre es helo


In [24]:
from langchain_core.prompts import ChatPromptTemplate

from langchain_core.output_parsers.json import SimpleJsonOutputParser

json_prompt = ChatPromptTemplate.from_messages(
    ("human", "Return a JSON Object with key as 'Answer' for the following question: {queston}")
)

chain = json_prompt | model

res = chain.invoke("What is the capital of USA")

print(res.content)

{"Answer":"Washington, D.C."}


In [28]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field



In [29]:
class Joke(BaseModel):
    setup: str = Field(description="Question to setup a joke")
    punchline: str  = Field(description = "Answer to resolve a joke")

In [37]:
parser = JsonOutputParser(pydantic_object=Joke)

prompt = PromptTemplate(
    template="Answer the user query ./ {format_instructions} \n {query}\n",
    input_variables=["query"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

chain =  prompt | model | parser

res = chain.invoke({"query":"Tell me a really really good joke in telugu"})

print(res)

{'setup': 'ఒకసారి నా ఫ్రెండ్: “నీవు ఏదైనా పనిలో ఎందుకంత త్వరగా అలసిపోతావు?” అన్నాడు.', 'punchline': 'నేను చెప్పాను: “ఎందుకంటే నేను వర్క్ హార్డ్ కాదు… వర్క్ హార్ట్\u200cతో చేస్తాను!”'}
